In [ ]:
import torch
import os
import torch.nn.functional as F
import torch.nn as nn
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import CLIPProcessor, CLIPModel, AutoTokenizer,AutoModel
import numpy as np

In [ ]:
TEACHER_ID = "openai/clip-vit-base-patch32"
STUDENT_ID = "Shushant/nepaliBERT"
#STUDENT_ID = "IRIISNEPAL/RoBERTa_Nepali_110M"
MAX_LENGTH = 77
BATCH_SIZE = 256
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
LOCAL_MODEL_PATH="/content/models"

In [ ]:
import os
import shutil
from concurrent.futures import ThreadPoolExecutor
from urllib.request import urlopen, Request
from tqdm.notebook import tqdm

def copy_worker(item_info):
    """Worker function to copy/download a single item only if it doesn't exist."""
    source, destination, item_name = item_info

    # Check if it already exists locally
    if os.path.exists(destination):
        return f"⏩ Skipped: {item_name} (Already exists)"

    # Handle Web URL download
    if source.startswith(("http://", "https://")):
        try:
            req = Request(source, headers={'User-Agent': 'Mozilla/5.0'})
            with urlopen(req) as response, open(destination, 'wb') as out_file:
                shutil.copyfileobj(response, out_file)
            return f"📥 URL Downloaded: {item_name}"
        except Exception as e:
            return f"❌ Failed URL: {item_name} ({e})"

    # Handle Google Drive / Local File system copy
    try:
        if os.path.isdir(source):
            shutil.copytree(source, destination)
            return f"✅ Folder copied: {item_name}"
        else:
            shutil.copy2(source, destination)
            return f"✅ File copied:   {item_name}"
    except Exception as e:
        return f"❌ Failed Copy: {item_name} ({e})"

def download_models_parallel(paths_to_download, local_path, max_workers=4):
    """Parallel downloader that accepts folders, single files, or file URLs."""
    if not os.path.exists(local_path):
        os.makedirs(local_path, exist_ok=True)
        print(f"📁 Created local directory: {local_path}")

    tasks = []

    # Scenario A: User passed a string path (Drive/Local path or URL)
    if isinstance(paths_to_download, str):
        # 1. Check if it's a web URL
        if paths_to_download.startswith(("http://", "https://")):
            item_name = os.path.basename(paths_to_download.split("?")[0])
            dest_path = os.path.join(local_path, item_name)
            tasks.append((paths_to_download, dest_path, item_name))
            print(f"📦 Found 1 URL link. Processing...")

        # 2. Check if it's an existing path on Drive/Local
        elif os.path.exists(paths_to_download):
            # If it's a single file (like your distillation_epoch_4_step_150000.pt)
            if os.path.isfile(paths_to_download):
                item_name = os.path.basename(paths_to_download)
                dest_path = os.path.join(local_path, item_name)
                tasks.append((paths_to_download, dest_path, item_name))
                print(f"📦 Found 1 single file path. Processing...")

            # If it's a directory containing multiple models
            elif os.path.isdir(paths_to_download):
                items = os.listdir(paths_to_download)
                tasks = [(os.path.join(paths_to_download, i), os.path.join(local_path, i), i) for i in items]
                print(f"📦 Found folder with {len(items)} items. Processing...")
        else:
            print(f"❌ Error: Path '{paths_to_download}' not found.")
            return

    # Scenario B: User passed a list of mixed paths/URLs
    elif isinstance(paths_to_download, list):
        for item in paths_to_download:
            item_name = os.path.basename(item.split("?")[0])
            dest_path = os.path.join(local_path, item_name)
            tasks.append((item, dest_path, item_name))
        print(f"📦 Processing list of {len(tasks)} custom items...")

    if not tasks:
        print("Empty path list provided.")
        return

    # Using ThreadPoolExecutor with a progress bar
    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(tqdm(executor.map(copy_worker, tasks), total=len(tasks)))

    # Print summary
    skipped = sum(1 for r in results if "Skipped" in r)
    success = sum(1 for r in results if "✅" in r or "📥" in r)
    failed = sum(1 for r in results if "❌" in r)

    print(f"\n✨ Done! {success} downloaded/copied, {skipped} skipped, {failed} failed.")

In [ ]:
model_path = '/content/drive/MyDrive/Nep-CLIP Datasets/Shushant-nepaliBERT distillation_epoch_4_step_150000.pt'
#model_path = '/content/drive/MyDrive/Nep-CLIP Datasets/RoBERTa_Nepali_110M distillation_epoch_4_step_150000.pt'

download_models_parallel(model_path,local_path=LOCAL_MODEL_PATH, max_workers=8)

#FLICKR-8K

In [ ]:
# 1. Download the zip
!wget "https://github.com/awsaf49/flickr-dataset/releases/download/v1.0/flickr8k.zip"

# 2. Unzip and clean up
!unzip -q flickr8k.zip -d ./flickr8k
!rm flickr8k.zip

print("Flickr8k dataset downloaded and extracted successfully.")

In [ ]:
# --- 3. DATA PREPARATION ---
# Download and unzip if not present
total_to_translate = None # Set to 40460 for full dataset

import os
if not os.path.exists('Flickr8k.token.txt'):
    !wget -q https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip
    !unzip -q Flickr8k_text.zip

In [ ]:
import pandas as pd

df_custom_flick = pd.read_csv("BT_Flickr8k_nepali_output.csv")

In [ ]:
# 2. Load split files
def load_split(file_path):
    with open(file_path, "r") as f:
        return [line.strip() for line in f]

train_imgs = load_split("Flickr_8k.trainImages.txt")
val_imgs = load_split("Flickr_8k.devImages.txt")
test_imgs = load_split("Flickr_8k.testImages.txt")

print("Train images:", len(train_imgs))
print("Val images:", len(val_imgs))
print("Test images:", len(test_imgs))

In [ ]:
flick_data_path = "flickr8k/Images/"

#FLICKR-30K

In [ ]:
# Install downloader
!pip install -q huggingface_hub

from huggingface_hub import hf_hub_download
import zipfile
import os

# Download images from a reliable mirror on Hugging Face
print("Downloading Flickr30k images (this may take a few minutes)...")
repo_id = "nlphuji/flickr30k"
file_path = hf_hub_download(repo_id=repo_id, filename="flickr30k-images.zip", repo_type="dataset")

# Unzip
print("Extracting...")
with zipfile.ZipFile(file_path, 'r') as zip_ref:
    zip_ref.extractall("./flickr30k")

print("Flickr30k images extracted to ./flickr30k")

In [ ]:
!wget -q https://cs.stanford.edu/people/karpathy/deepimagesent/caption_datasets.zip
!unzip -q caption_datasets.zip
!ls *.json  # should see dataset_flickr30k.json

In [ ]:
import pandas as pd

df_custom_flick = pd.read_csv("BT_Flickr30k_nepali_output.csv")

In [ ]:
import json

with open('dataset_flickr30k.json', 'r') as f:
    data = json.load(f)

train, val, test = [], [], []

for img in data['images']:
    entry = {
        'filename': img['filename'],
        'imgid':    img['imgid'],
        'captions': [c['raw'] for c in img['sentences']]
    }
    if img['split'] == 'train':
        train.append(entry)
    elif img['split'] == 'val':
        val.append(entry)
    elif img['split'] == 'test':
        test.append(entry)

print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")

In [ ]:
# Extract the actual filenames (Image IDs) for the test set
test_imgs = [img['filename'] for img in test]

# Display results
print(f"Total Test Images: {len(test_imgs)}")
print(f"First 5 Image IDs: {test_imgs[:5]}")

In [ ]:
flick_data_path = "flickr30k/flickr30k-images/"

#EVALUATION

In [ ]:
clip_processor = CLIPProcessor.from_pretrained(TEACHER_ID)
s_tokenizer = AutoTokenizer.from_pretrained(STUDENT_ID)
clip_model = CLIPModel.from_pretrained(TEACHER_ID)

In [ ]:
class NepaliTextEncoder(nn.Module):
  def __init__(self, student_id):
      super().__init__()
      self.student = AutoModel.from_pretrained(student_id)
      self.student.eval()
      self.projection = nn.Linear(768, 512)

  def get_nepali_text_embeddings(self, student_inputs):
    text_embeddings = self.student(**student_inputs)
    return text_embeddings.pooler_output

  def forward(self,student_inputs):
    nepali_embeddings = self.get_nepali_text_embeddings(student_inputs=student_inputs)
    student_projected_embeddings = self.projection(nepali_embeddings)
    return  student_projected_embeddings

In [ ]:
from PIL import Image
class FlickrkEvalDataset(Dataset):
    def __init__(self, image_paths, captions, image_processor, tokenizer, max_length=77):
        print(f"Pre-processing {len(image_paths)} images and {len(captions)} captions...")

        # 1. Pre-process Images
        processed_images = []
        for path in image_paths:
            img = Image.open(path).convert("RGB")
            encoding = image_processor(images=img, return_tensors="pt")
            processed_images.append(encoding.pixel_values.squeeze(0))

        self.pixel_values = torch.stack(processed_images, dim=0)

        # 2. Pre-tokenize Captions
        self.text_inputs = tokenizer(
            captions,
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

    def __len__(self):
        # Return length of captions (usually the primary index for datasets)
        return len(self.text_inputs['input_ids'])

    def get_all_data(self):
        """Helper to get everything for run_report"""
        return self.pixel_values, self.text_inputs

In [ ]:
# Ensure captions align with the images (5 captions each)
import re
df_custom_flick['image_id_update'] = df_custom_flick['image_id'].str.replace(r'(\.jpg).*', r'\1', regex=True)

caption_dict = df_custom_flick.groupby('image_id_update')['translated_caption_np'].apply(list).to_dict()

test_images = []
test_captions = []

for img in test_imgs:
    # Get the 5 captions for this specific image
    caps = caption_dict.get(img, [])
    test_images.append(f"{flick_data_path}{img}")
    if len(caps) < 5:
      print(img,len(caps))
    for c in caps:
        test_captions.append(c)

In [ ]:
# 1. Initialize Dataset
dataset = FlickrkEvalDataset(test_images, test_captions, clip_processor, s_tokenizer)

# 2. Get processed tensors
image_values, text_inputs = dataset.get_all_data()

In [ ]:
def get_image_features(clip_model,image_values,batch_size=256,device="cuda"):
  img_embs = []
  clip_model.to(device).eval()
  with torch.no_grad():
    for i in tqdm(range(0, len(image_values), batch_size), desc="Encoding Images"):
      batch_pixels = image_values[i : i + batch_size].to(device)
      img_f = clip_model.get_image_features(pixel_values=batch_pixels)
      # Handle potential pooler_output or raw output depending on CLIP version
      if hasattr(img_f, 'pooler_output'):
          img_f = img_f.pooler_output

      img_f = img_f / img_f.norm(dim=-1, keepdim=True)
      img_embs.append(img_f.cpu())
  return torch.cat(img_embs, dim=0).numpy() # [N_images, D]

In [ ]:
def get_text_features(encoder,encoder_name,text_inputs,batch_size=256,device="cuda"):
  # --- 2. Loop through each Nepali Text Encoder ---
  encoder.to(device).eval()
  txt_embs = []

  num_texts = text_inputs['input_ids'].shape[0]
  with torch.no_grad():
      for i in tqdm(range(0, num_texts, batch_size), desc=f"Encoding Text ({encoder_name})"):
          batch_text = {k: v[i : i + batch_size].to(device) for k, v in text_inputs.items()}

          txt_f = encoder(batch_text)
          txt_f = txt_f / txt_f.norm(dim=-1, keepdim=True)
          txt_embs.append(txt_f.cpu())

  return torch.cat(txt_embs, dim=0).numpy() # [N_texts, D]

In [ ]:
def calculate_ap(found_ranks, total_relevant):
    """
    Calculates Average Precision for a single query.
    found_ranks: 0-indexed positions of all correct items in the sorted list.
    total_relevant: Total number of ground-truth items (e.g., 5 for Flickr8k I2T).
    """
    if len(found_ranks) == 0:
        return 0.0

    # Sort ranks to ensure we calculate P@k in order (1st found, 2nd found...)
    sort_ranks = np.sort(found_ranks) + 1
    precision_sum = 0.0
    for i, r in enumerate(sort_ranks):
        # (i + 1) is the count of relevant items found so far
        precision_sum += (i + 1) / r
    return precision_sum / total_relevant

In [ ]:
def calculate_mar(found_ranks, total_relevant, k=10):
    """
    Calculates Average Recall @ K for a single query.
    found_ranks: 0-indexed positions of all correct items.
    total_relevant: Total possible relevant items (5 for Flickr8k I2T).
    k: The cutoff for recall.
    """
    # Count how many of the found_ranks are within the top K
    hits_at_k = np.sum(found_ranks < k)
    return hits_at_k / total_relevant

In [ ]:
def calculate_maf1(found_ranks, total_relevant, k=10):
    """
    Calculates F1 @ K for a single query.
    found_ranks: 0-indexed positions of all correct items.
    total_relevant: Total possible relevant items (5 for Flickr8k I2T).
    k: The cutoff for precision and recall.
    """
    hits_at_k = np.sum(found_ranks < k)

    precision_at_k = hits_at_k / k
    recall_at_k = hits_at_k / total_relevant

    if (precision_at_k + recall_at_k) == 0:
        return 0.0

    f1 = 2 * (precision_at_k * recall_at_k) / (precision_at_k + recall_at_k)
    return f1

In [ ]:
def compute_metrics(similarity, query_type='i2t', captions_per_image=5):
    """
    I2T: Row-wise search (Is the correct caption in the top K for this image?)
    T2I: Column-wise search (Is the correct image in the top K for this caption?)
    """
    aps, ars, af1s = [], [], []
    if query_type == 'i2t':
        # [N_images, N_texts] -> Axis 0 is images
        num_queries = similarity.shape[0]
        ranks = []
        for i in range(num_queries):
            scores = similarity[i]
            inds = np.argsort(scores)[::-1]
            gt_indices = np.arange(i * captions_per_image, (i + 1) * captions_per_image)

            # Find all positions where the ground truth captions appear
            all_found_ranks = np.where(np.isin(inds, gt_indices))[0]

            # Use the best (first) rank for standard R@K metrics
            best_rank = all_found_ranks[0]
            ranks.append(best_rank)

            # Calculate AP using the external function
            aps.append(calculate_ap(all_found_ranks, captions_per_image))
            ars.append(calculate_mar(all_found_ranks, captions_per_image, k=10))
            af1s.append(calculate_maf1(all_found_ranks, captions_per_image, k=10))
    else:
        # [N_images, N_texts] -> Axis 1 is texts
        num_queries = similarity.shape[1]
        ranks = []
        for j in range(num_queries):
            scores = similarity[:, j]
            inds = np.argsort(scores)[::-1]
            gt_image_index = j // captions_per_image

            rank = np.where(inds == gt_image_index)[0][0]
            ranks.append(rank)

            # For T2I (1 relevant item), AP is just 1/rank
            aps.append(calculate_ap(np.array([rank]), 1))
            ars.append(calculate_mar(np.array([rank]), 1, k=10))
            af1s.append(calculate_maf1(np.array([rank]), 1, k=10))
    ranks = np.array(ranks)
    metrics = {
        "R@1":  float(np.mean(ranks < 1) * 100),
        "R@5":  float(np.mean(ranks < 5) * 100),
        "R@10": float(np.mean(ranks < 10) * 100),
        "MedR": float(np.median(ranks) + 1),
        "MAP":  float(np.mean(aps) * 100),
        "MAR":  float(np.mean(ars) * 100),
        "MAF1": float(np.mean(af1s) * 100)
    }

    r_indexed = ranks + 1
    for k in range(1, 11):
        mrr_k = np.where(r_indexed <= k, 1.0 / r_indexed, 0.0).mean()
        metrics[f"MRR@{k}"] = float(mrr_k)

    return metrics

In [ ]:
def run_full_report(text_encoders_dict, clip_model, image_values, text_inputs, batch_size=128):
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

  # 1. Get Image Embeddings once
  img_emb = get_image_features(clip_model, image_values, batch_size, device)

  results = {}

  # 2. Loop through each Nepali Text Encoder
  for name, encoder in text_encoders_dict.items():
    print(f"\nEvaluating: {name}")

    # Get Text Embeddings
    txt_emb = get_text_features(encoder,name, text_inputs, batch_size, device)

    # 3. Compute Similarity
    similarity = img_emb @ txt_emb.T

    # 4. Store metrics
    results[name] = {
        "txt_emb":txt_emb,
        "I2T": compute_metrics(similarity, query_type='i2t'),
        "T2I": compute_metrics(similarity, query_type='t2i')
    }

    # Memory management
    encoder.cpu()
    torch.cuda.empty_cache()
  return {
      "results":results,
      "img_emb":img_emb
      }


In [ ]:
import os
import torch

def get_all_models_with_loaded_instance(model_folder, STUDENT_ID):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # List all checkpoint files in folder 'x'
    checkpoint_files = [f for f in os.listdir(model_folder) if f.endswith(('.pt', '.pth'))]

    encoders_dict = {}

    for ckpt_name in checkpoint_files:
      print(f"Loading weights from: {ckpt_name}")
      ckpt_path = os.path.join(model_folder, ckpt_name)

      # 1. Initialize a fresh instance of your encoder
      encoder = NepaliTextEncoder(STUDENT_ID)

      # 2. Load the checkpoint
      ckpt = torch.load(ckpt_path, map_location=device)

      # 3. Load state dicts into the sub-modules
      encoder.student.load_state_dict(ckpt["student_state_dict"])
      encoder.projection.load_state_dict(ckpt["projection_state_dict"])

      # Store using the filename (or a pretty name) as the key
      encoders_dict[ckpt_name] = encoder

    return encoders_dict

In [ ]:
text_encoders_dict = get_all_models_with_loaded_instance(model_folder=LOCAL_MODEL_PATH,STUDENT_ID=STUDENT_ID)


In [ ]:
#flick8k
complete_reports_8K = run_full_report(
        text_encoders_dict=text_encoders_dict,
        clip_model=clip_model,
        image_values=image_values,
        text_inputs=text_inputs,
        batch_size=BATCH_SIZE
    )

In [ ]:
#flick30k
complete_reports_30K = run_full_report(
        text_encoders_dict=text_encoders_dict,
        clip_model=clip_model,
        image_values=image_values,
        text_inputs=text_inputs,
        batch_size=BATCH_SIZE
    )

In [ ]:
import re
def get_numeric_step(name):
    """Extracts numeric step from string 'step_1234' (expects a string)."""
    match = re.search(r'step_(\d+)', name)
    return int(match.group(1)) if match else 0

In [ ]:
def log_report(reports):
  # Header updated with MAP, MAR, MAF1, and MRR
  header = f"{'Model Checkpoint':<30} | {'I2T (R@1 / R@5 / R@10  |  M  |  MAP  |  MAR  |  MF1  |  MRR)':<80} | {'T2I (R@1 / R@5 / R@10  |  M  |  MAP  |  MAR  |  MF1  |  MRR)':<80}"
  print(header)
  print("-" * len(header))

  for model, metrics in sorted(reports.items(), key=lambda x: get_numeric_step(x[0])):
      i2t = metrics['I2T']
      t2i = metrics['T2I']

      # Formatting strings to include MAF1 (MF1)
      i2t_str = (f"{i2t['R@1']:>4.2f}/{i2t['R@5']:>4.2f}/{i2t['R@10']:>4.2f} | "
                f"M:{i2t['MedR']:<2.0f} | {i2t['MAP']:>5.2f} | {i2t['MAR']:>5.2f} | "
                f"{i2t['MAF1']:>5.2f} | {i2t['MRR@10']*100:>5.2f}")

      t2i_str = (f"{t2i['R@1']:>4.2f}/{t2i['R@5']:>4.2f}/{t2i['R@10']:>4.2f} | "
                f"M:{t2i['MedR']:<2.0f} | {t2i['MAP']:>5.2f} | {t2i['MAR']:>5.2f} | "
                f"{t2i['MAF1']:>5.2f} | {t2i['MRR@10']*100:>5.2f}")

      print(f"{model:<30} | {i2t_str:<60} | {t2i_str:<60}")

In [ ]:
#flickr8k
reports = complete_reports_8K["results"]
log_report(reports)

In [ ]:
#flickr30K
reports = complete_reports_30K["results"]
log_report(reports)